## Bonus Exercise: Bikeshare

Here is your chance to apply what you have learned to some real data. We will be using a bike sharing dataset, but a very different one from last class

Here you’re going to be working with publicly available bike data from the Bay Area Bike Share portal, specifically analyzing the 2017 year of data. First we will download the data and read it into spark, we will also rename some of the columns to meet the requirements of graphframes.

In [1]:
#Checking the installed Java version
!java -version
!pip install "pyspark==3.5.0" 
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

!java -version

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-124.04, mixed mode, sharing)
Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 https://download.docker.com/linux/ubuntu noble InRelease                 
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:5 https://archive.ubuntu.com/ubuntu noble InRelease                        
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:7 https://archive.ubuntu.com/ubuntu noble-updates InRelease                
Hit:8 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 http://deb.wakemeops.com/wakemeops stable InRelease                      
Hit:10 https://archive.ubuntu.com/ubuntu noble-backports InRelease             
Hit:11 https://cloud.archive.ubuntu.com

In [2]:
%pip install graphframes-py==0.10.0

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GraphFramesWithSpark4") \
    .config("spark.jars.packages", "io.graphframes:graphframes-spark3_2.12:0.10.0") \
    .getOrCreate()

print(f"spark version: {spark.version}")
print("spark session created with graphframes package specified!")

:: loading settings :: url = jar:file:/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/zeus/.ivy2/cache
The jars for the packages stored in: /home/zeus/.ivy2/jars
io.graphframes#graphframes-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f2c6eb13-bc57-4e26-9342-3ef1cb5661ad;1.0
	confs: [default]
	found io.graphframes#graphframes-spark3_2.12;0.10.0 in central
	found io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 in central
:: resolution report :: resolve 223ms :: artifacts dl 8ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark3_2.12;0.10.0 from central in [default]
	io.graphframes#graphframes-spark3_2.12;0.10.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||  

spark version: 3.5.0
spark session created with graphframes package specified!


In [4]:
#import the package we just installed
from graphframes import *
#import data types - All data types of Spark SQL are located in the package of pyspark.sql.types
from pyspark.sql.types import *
#row can be used to create a row object by using named arguments
from pyspark.sql import Row
from pyspark.sql.functions import col


In [5]:
!wget -O /teamspace/studios/this_studio/week12/bikes/201508_trip_data.csv https://raw.githubusercontent.com/udacity/data-analyst/master/projects/bike_sharing/201508_trip_data.csv

--2025-11-26 17:58:55--  https://raw.githubusercontent.com/udacity/data-analyst/master/projects/bike_sharing/201508_trip_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 43012650 (41M) [text/plain]
Saving to: ‘/teamspace/studios/this_studio/week12/bikes/201508_trip_data.csv’

/teamspace/studios/ 100%[===================>]  41.02M   136MB/s    in 0.3s    

2025-11-26 17:58:56 (136 MB/s) - ‘/teamspace/studios/this_studio/week12/bikes/201508_trip_data.csv’ saved [43012650/43012650]



In [6]:
!wget -O /teamspace/studios/this_studio/week12/bikes/201508_station_data.csv https://raw.githubusercontent.com/udacity/data-analyst/master/projects/bike_sharing/201508_station_data.csv

--2025-11-26 17:58:56--  https://raw.githubusercontent.com/udacity/data-analyst/master/projects/bike_sharing/201508_station_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5272 (5.1K) [text/plain]
Saving to: ‘/teamspace/studios/this_studio/week12/bikes/201508_station_data.csv’

/teamspace/studios/ 100%[===================>]   5.15K  --.-KB/s    in 0s      

2025-11-26 17:58:56 (26.8 MB/s) - ‘/teamspace/studios/this_studio/week12/bikes/201508_station_data.csv’ saved [5272/5272]



In [7]:
tripDF = spark.read.csv('/teamspace/studios/this_studio/week12/bikes/201508_trip_data.csv',header=True, inferSchema = True)
tripDF.printSchema()

root
 |-- Trip ID: integer (nullable = true)
 |-- Duration: integer (nullable = true)
 |-- Start Date: string (nullable = true)
 |-- Start Station: string (nullable = true)
 |-- Start Terminal: integer (nullable = true)
 |-- End Date: string (nullable = true)
 |-- End Station: string (nullable = true)
 |-- End Terminal: integer (nullable = true)
 |-- Bike #: integer (nullable = true)
 |-- Subscriber Type: string (nullable = true)
 |-- Zip Code: string (nullable = true)



In [8]:
stationDF = spark.read.csv('/teamspace/studios/this_studio/week12/bikes/201508_station_data.csv',header=True, inferSchema = True)
stationDF.printSchema()

root
 |-- station_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dockcount: integer (nullable = true)
 |-- landmark: string (nullable = true)
 |-- installation: string (nullable = true)



In [9]:
stationVertices = stationDF.withColumnRenamed("name", "id").distinct()
tripEdges = tripDF.withColumnRenamed("Start Station", "src").withColumnRenamed("End Station", "dst")

In [10]:
tripEdges.show(10, False)

+-------+--------+---------------+---------------------------------------------+--------------+---------------+---------------------------------------------+------------+------+---------------+--------+
|Trip ID|Duration|Start Date     |src                                          |Start Terminal|End Date       |dst                                          |End Terminal|Bike #|Subscriber Type|Zip Code|
+-------+--------+---------------+---------------------------------------------+--------------+---------------+---------------------------------------------+------------+------+---------------+--------+
|913460 |765     |8/31/2015 23:26|Harry Bridges Plaza (Ferry Building)         |50            |8/31/2015 23:39|San Francisco Caltrain (Townsend at 4th)     |70          |288   |Subscriber     |2139    |
|913459 |1036    |8/31/2015 23:11|San Antonio Shopping Center                  |31            |8/31/2015 23:28|Mountain View City Hall                      |27          |35    |Subscriber 

In [11]:
stationVertices.show(10, False)

+----------+-------------------------------------+---------+-----------+---------+-------------+------------+
|station_id|id                                   |lat      |long       |dockcount|landmark     |installation|
+----------+-------------------------------------+---------+-----------+---------+-------------+------------+
|46        |Washington at Kearney                |37.795425|-122.404767|15       |San Francisco|8/19/2013   |
|41        |Clay at Battery                      |37.795001|-122.39997 |15       |San Francisco|8/19/2013   |
|33        |Rengstorff Avenue / California Street|37.400241|-122.099076|15       |Mountain View|8/16/2013   |
|29        |San Antonio Caltrain Station         |37.40694 |-122.106758|23       |Mountain View|8/15/2013   |
|42        |Davis at Jackson                     |37.79728 |-122.398436|15       |San Francisco|8/19/2013   |
|13        |St James Park                        |37.339301|-121.889937|15       |San Jose     |8/6/2013    |
|54       

Alright you now have the data formatted as you need to perform analysis. Take a look to make sure you understand what the data is. Basically you have bike stations (verticies) and trips between the stations (edges). 

Hint: in the cell above I used 'False' in the `show()`, this will be useful for you to prevent truncating station names. The highest ranked station should be San Jose Diridon Caltrain Station.

#####Create a graphframe and perform a pagerank to find the most import stations (make sure to set the maxIterations to 5).

In [12]:
#take the vertices and edges DataFrames and returns a GraphFrames object.
gbs = GraphFrame(stationVertices, tripEdges)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


In [13]:
#show the edges
gbs.edges.show(10, False)

+-------+--------+---------------+---------------------------------------------+--------------+---------------+---------------------------------------------+------------+------+---------------+--------+
|Trip ID|Duration|Start Date     |src                                          |Start Terminal|End Date       |dst                                          |End Terminal|Bike #|Subscriber Type|Zip Code|
+-------+--------+---------------+---------------------------------------------+--------------+---------------+---------------------------------------------+------------+------+---------------+--------+
|913460 |765     |8/31/2015 23:26|Harry Bridges Plaza (Ferry Building)         |50            |8/31/2015 23:39|San Francisco Caltrain (Townsend at 4th)     |70          |288   |Subscriber     |2139    |
|913459 |1036    |8/31/2015 23:11|San Antonio Shopping Center                  |31            |8/31/2015 23:28|Mountain View City Hall                      |27          |35    |Subscriber 

In [14]:
#Run the pageRank()
pageRanks = gbs.pageRank(resetProbability=0.15, maxIter = 5)

25/11/26 17:59:13 WARN BlockManager: Block rdd_114_0 already exists on this machine; not re-adding it
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [15]:
#Show() the vertices, ranked
pageRanks.vertices.orderBy(pageRanks.vertices.pagerank,ascending=False).show(10,False)

+----------+----------------------------------------+---------+-----------+---------+-------------+------------+------------------+
|station_id|id                                      |lat      |long       |dockcount|landmark     |installation|pagerank          |
+----------+----------------------------------------+---------+-----------+---------+-------------+------------+------------------+
|2         |San Jose Diridon Caltrain Station       |37.329732|-121.901782|27       |San Jose     |8/6/2013    |4.084086216116666 |
|70        |San Francisco Caltrain (Townsend at 4th)|37.776617|-122.39526 |19       |San Francisco|8/23/2013   |3.351543505799    |
|28        |Mountain View Caltrain Station          |37.394358|-122.076713|23       |Mountain View|8/15/2013   |2.5531183688195247|
|22        |Redwood City Caltrain Station           |37.486078|-122.232089|25       |Redwood City |8/15/2013   |2.4734490963466995|
|69        |San Francisco Caltrain 2 (330 Townsend) |37.7766  |-122.39547 |2

One question is: what are the most common trip paths? You can do this by performing a grouping operator and adding the edge counts together, think of how we used sparkSQL to implement SQL like queries last week. Remember the list of edges is just a dataframe so you can use SQL queries to find a result like this. In case you have forgotten this link has some information on groupBy:

https://sparkbyexamples.com/spark/using-groupby-on-dataframe/

The most common trip should be:

San Francisco Caltrain 2 (330 Townsend)  ->    Townsend at 7th.

In [16]:
tripEdgesSum = tripEdges.groupBy(tripEdges.src, tripEdges.dst).count()
tripEdgesSum.orderBy("count",ascending=False).show(10, False)

+---------------------------------------------+----------------------------------------+-----+
|src                                          |dst                                     |count|
+---------------------------------------------+----------------------------------------+-----+
|San Francisco Caltrain 2 (330 Townsend)      |Townsend at 7th                         |3748 |
|Harry Bridges Plaza (Ferry Building)         |Embarcadero at Sansome                  |3145 |
|2nd at Townsend                              |Harry Bridges Plaza (Ferry Building)    |2973 |
|Townsend at 7th                              |San Francisco Caltrain 2 (330 Townsend) |2734 |
|Harry Bridges Plaza (Ferry Building)         |2nd at Townsend                         |2640 |
|Embarcadero at Folsom                        |San Francisco Caltrain (Townsend at 4th)|2439 |
|Steuart at Market                            |2nd at Townsend                         |2356 |
|Embarcadero at Sansome                       |Ste

In [17]:
#Create a gbsUnique graphframe
gbsUnique = GraphFrame(stationVertices, tripEdgesSum)

Remember that in this instance you’ve got a directed graph. That means that your trips are directional - from one location to another. Therefore you get access to a wealth of analysis that you can use. You can find the number of trips that go into a specific station and leave from a specific station.

One interesting question you could ask is what is the station with the highest ratio of in degrees to out degrees. As in, what station acts as a pure trip sink. A station where trips end at but rarely start from. This station will end up with a lot of excess bikes that will need to be re-distributed.

In [18]:
from pyspark.sql.functions import col
gbsOut = gbs.outDegrees
gbsIn = gbs.inDegrees
# Note you may find it easier to use q SQL query here if that is what you are used to
gbsJoin = gbsIn.join(gbsOut, ["id"])
gbsDeg = gbsJoin.withColumn("Ratio", col("inDegree") / col("outDegree"))

In [19]:
gbsJoin.printSchema()

root
 |-- id: string (nullable = true)
 |-- inDegree: integer (nullable = false)
 |-- outDegree: integer (nullable = false)



In [20]:
gbsDeg.orderBy("Ratio",ascending=False).show(10)

+--------------------+--------+---------+------------------+
|                  id|inDegree|outDegree|             Ratio|
+--------------------+--------+---------+------------------+
|Redwood City Medi...|     230|      150|1.5333333333333334|
|San Mateo County ...|     187|      127|1.4724409448818898|
|SJSU 4th at San C...|     647|      475|1.3621052631578947|
|San Francisco Cal...|   34810|    26304|1.3233728710462287|
|Washington at Kearny|    3481|     2660|1.3086466165413533|
|Paseo de San Antonio|    1073|      856|1.2535046728971964|
|California Ave Ca...|     496|      400|              1.24|
|   Franklin at Maple|     100|       81|1.2345679012345678|
|Embarcadero at Va...|    6146|     5037|1.2201707365495336|
|   Market at Sansome|   13916|    11431|1.2173913043478262|
+--------------------+--------+---------+------------------+
only showing top 10 rows



Consider the station with the highest ratio of intrips to outtrips, find all stations connected by 1 or 2 trips to this station using motifs.

In [21]:
#search for pairs of vertices a,b connected by edges in both directions
motifsBS = gbsUnique.find("(a)-[]->(b); (b)-[]->(c)").dropDuplicates()

In [22]:
# Find stations connected by two trips
filteredBS = motifsBS.filter("a.id = 'Redwood City Medical Center'")
filteredBS.show(10)

+--------------------+--------------------+--------------------+
|                   a|                   b|                   c|
+--------------------+--------------------+--------------------+
|{26, Redwood City...|{25, Stanford in ...|{24, Redwood City...|
|{26, Redwood City...|{21, Franklin at ...|{34, Palo Alto Ca...|
|{26, Redwood City...|{22, Redwood City...|{38, Park at Oliv...|
|{26, Redwood City...|{26, Redwood City...|{24, Redwood City...|
|{26, Redwood City...|{83, Mezes Park, ...|{36, California A...|
|{26, Redwood City...|{24, Redwood City...|{24, Redwood City...|
|{26, Redwood City...|{26, Redwood City...|{26, Redwood City...|
|{26, Redwood City...|{83, Mezes Park, ...|{34, Palo Alto Ca...|
|{26, Redwood City...|{24, Redwood City...|{25, Stanford in ...|
|{26, Redwood City...|{25, Stanford in ...|{23, San Mateo Co...|
+--------------------+--------------------+--------------------+
only showing top 10 rows



In [23]:
#store the intermediate computation so they can be reused in subsequent actions.
filteredBS.persist()

DataFrame[a: struct<station_id:int,id:string,lat:double,long:double,dockcount:int,landmark:string,installation:string>, b: struct<station_id:int,id:string,lat:double,long:double,dockcount:int,landmark:string,installation:string>, c: struct<station_id:int,id:string,lat:double,long:double,dockcount:int,landmark:string,installation:string>]

In [24]:
#count the number os connections
filteredBS.count()

64

In [25]:
# Find stations connected by one trip
motifsBS1 = gbsUnique.find("(a)-[]->(b)").dropDuplicates()
motifsBS1.filter("a.id = 'Redwood City Medical Center'").count()

7

## Some More Exercises

In [26]:
import pyspark.sql.functions as F

### Exercise 1 – Transfer hubs in the bike network

In this exercise, you will identify the key “transfer hubs” in the bike-share network. These are stations that are structurally important in the graph and also handle a large volume of trips. Use graph measures of centrality together with trip-level usage information to build a composite “hub” score, then find the top 5 stations according to this score. Interpret what makes these stations important in the system and whether they correspond to your intuition about major transfer points.


In [27]:
# Solution for Exercise 1

# 1. PageRank
pr = gbsUnique.pageRank(resetProbability=0.15, maxIter=10)
v_pr = pr.vertices.select("id", "pagerank", "landmark", "dockcount")

# 2. Degrees
deg = gbsUnique.degrees  # columns: id, degree

# 3. Outgoing trip counts per station
trip_out = (
    gbsUnique.edges
        .groupBy("src")
        .agg(F.count("*").alias("outgoing_trip_count"))
        .withColumnRenamed("src", "id")
)

# 4. Combine centrality and usage into a hub score
hubs = (
    v_pr
        .join(deg, on="id", how="left")
        .join(trip_out, on="id", how="left")
        .fillna({"degree": 0, "outgoing_trip_count": 0})
        .withColumn(
            "hub_score",
            F.col("pagerank")
            * F.log1p(F.col("degree"))
            * F.log1p(F.col("outgoing_trip_count"))
        )
)

# 5. Top 5 transfer hubs
top5_hubs = (
    hubs
        .orderBy(F.col("hub_score").desc())
        .limit(5)
)

top5_hubs.select(
    "id",
    "landmark",
    "dockcount",
    "pagerank",
    "degree",
    "outgoing_trip_count",
    "hub_score",
).show(truncate=False)


25/11/26 17:59:22 WARN CacheManager: Asked to cache already cached data.


+----------------------------------------+-------------+---------+------------------+------+-------------------+------------------+
|id                                      |landmark     |dockcount|pagerank          |degree|outgoing_trip_count|hub_score         |
+----------------------------------------+-------------+---------+------------------+------+-------------------+------------------+
|San Francisco Caltrain (Townsend at 4th)|San Francisco|19       |1.188219541479926 |73    |36                 |18.466863413294256|
|Market at 4th                           |San Francisco|19       |1.1576256232898101|73    |36                 |17.99138418679484 |
|Howard at 2nd                           |San Francisco|19       |1.1141490187539218|71    |35                 |17.07490641947976 |
|Powell Street BART                      |San Francisco|19       |1.0488105597941761|71    |36                 |16.1964566536078  |
|Beale at Market                         |San Francisco|19       |1.04881055

### Exercise 2 – Community structure and internal versus external flows

In this exercise, you will detect communities of stations in the bike network and use these communities to study travel patterns. First, identify groups of stations that form communities in the graph. Then, use the trip data to distinguish between “internal” trips that stay within the same community and “external” trips that cross from one community to another. Compare communities that are mostly self-contained with those that act as gateways, and discuss what this reveals about the organisation of mobility in the system.


In [28]:
# Solution for Exercise 2

# 1. Label propagation
lp = gbsUnique.labelPropagation(maxIter=5)  # columns: id, label, plus vertex attributes

# 2. Attach labels to src and dst
src_labels = lp.select(F.col("id").alias("src"), F.col("label").alias("src_label"))
dst_labels = lp.select(F.col("id").alias("dst"), F.col("label").alias("dst_label"))

edges_with_labels = (
    gbsUnique.edges
        .join(src_labels, on="src", how="left")
        .join(dst_labels, on="dst", how="left")
)

# 3. Internal vs external trips per source community
flow_stats = (
    edges_with_labels
        .withColumn("is_internal", F.col("src_label") == F.col("dst_label"))
        .groupBy("src_label")
        .agg(
            F.sum(F.when(F.col("is_internal"), 1).otherwise(0)).alias("internal_trips"),
            F.sum(F.when(F.col("is_internal"), 0).otherwise(1)).alias("external_trips"),
        )
        .withColumn(
            "total_trips",
            F.col("internal_trips") + F.col("external_trips"),
        )
        .withColumn(
            "internal_share",
            F.when(F.col("total_trips") > 0,
                   F.col("internal_trips") / F.col("total_trips"))
             .otherwise(F.lit(None)),
        )
)

# 4. Top self-contained and gateway communities
self_contained = flow_stats.orderBy(F.col("internal_share").desc()).limit(5)
gateway = flow_stats.orderBy(F.col("internal_share").asc()).limit(5)

print("Top 5 self-contained communities (by internal share):")
self_contained.show()

print("Top 5 gateway communities (by internal share):")
gateway.show()


Top 5 self-contained communities (by internal share):
+---------+--------------+--------------+-----------+------------------+
|src_label|internal_trips|external_trips|total_trips|    internal_share|
+---------+--------------+--------------+-----------+------------------+
|       59|           255|             4|        259|0.9845559845559846|
|       23|          1089|            70|       1159|0.9396031061259706|
|       32|           119|            21|        140|              0.85|
|       41|            46|            18|         64|           0.71875|
|     NULL|             0|            70|         70|               0.0|
+---------+--------------+--------------+-----------+------------------+

Top 5 gateway communities (by internal share):


+---------+--------------+--------------+-----------+------------------+
|src_label|internal_trips|external_trips|total_trips|    internal_share|
+---------+--------------+--------------+-----------+------------------+
|     NULL|             0|            70|         70|               0.0|
|       41|            46|            18|         64|           0.71875|
|       32|           119|            21|        140|              0.85|
|       23|          1089|            70|       1159|0.9396031061259706|
|       59|           255|             4|        259|0.9845559845559846|
+---------+--------------+--------------+-----------+------------------+



### Exercise 3 – The giant component and peripheral sub-networks

In this exercise, you will explore global connectivity in the bike network. Start by finding the largest connected component, often called the “giant component,” and separate it from the smaller components. Compare these two sets of stations in terms of their number of docks, their coverage in the network, and their trip activity. Use your results to describe how much of the system is integrated into a single large network and what characterises the more peripheral or isolated sub-networks.


In [29]:
# Solution for Exercise 3
sc = spark.sparkContext
sc.setCheckpointDir("/tmp/graphframes-bikes-connected-components")

# 1. Connected components (returns a DataFrame)
components = gbsUnique.connectedComponents()  # columns: id, component, plus vertex attributes

# 2. Component sizes and giant component id
component_sizes = components.groupBy("component").agg(F.count("*").alias("num_vertices"))
giant_component_id = component_sizes.orderBy(F.col("num_vertices").desc()).first()["component"]

# 3. Mark group on vertices (giant vs small)
vertices_with_group = (
    components
        .withColumn(
            "group",
            F.when(F.col("component") == giant_component_id, F.lit("giant"))
             .otherwise(F.lit("small"))
        )
        .select("id", "group", "dockcount", "landmark")
)

print("Stations in giant component:", vertices_with_group.filter("group = 'giant'").count())
print("Stations in smaller components:", vertices_with_group.filter("group = 'small'").count())

# 4. Average dock count per group
group_dock = (
    vertices_with_group
        .groupBy("group")
        .agg(F.avg("dockcount").alias("avg_dockcount"))
)

print("Average dock count by group:")
group_dock.show()

# 5. Usage by group (based on source station)
edges_with_group = (
    gbsUnique.edges
        .join(
            vertices_with_group.select("id", "group"),
            gbsUnique.edges.src == vertices_with_group.id,
            how="left",
        )
        .withColumnRenamed("group", "src_group")
        .drop(vertices_with_group.id)
)

group_usage = (
    edges_with_group
        .groupBy("src_group")
        .agg(
            F.countDistinct("src").alias("num_source_stations"),
            F.count("*").alias("num_trips"),
        )
        .filter(F.col("src_group").isNotNull())
)



print("Usage by group (giant vs small):")
group_usage.show()


Stations in giant component: 68
Stations in smaller components: 2
Average dock count by group:
+-----+------------------+
|group|     avg_dockcount|
+-----+------------------+
|giant|17.676470588235293|
|small|              17.0|
+-----+------------------+

Usage by group (giant vs small):
+---------+-------------------+---------+
|src_group|num_source_stations|num_trips|
+---------+-------------------+---------+
|    giant|                 68|     1622|
+---------+-------------------+---------+



### Exercise 6 – Sources, sinks, and commuting patterns

In this exercise, you will characterise stations by how unbalanced their flows are. Some stations behave like “sources,” where many trips start but relatively few end, while others behave like “sinks,” receiving more trips than they send. Using the directed bike network, quantify net inflow and outflow for each station and use this to classify their commuting role. Relate these roles to station attributes such as landmark (city), installation date, or dock count, and discuss whether the main sources and sinks correspond to residential areas, business districts, or major transit nodes.


In [30]:

# Outgoing trips per station (sum of counts from src)
outgoing = (
    gbsUnique.edges
        .groupBy("src")
        .agg(F.sum("count").alias("outgoing_trips"))
        .withColumnRenamed("src", "id")
)

# Incoming trips per station (sum of counts to dst)
incoming = (
    gbsUnique.edges
        .groupBy("dst")
        .agg(F.sum("count").alias("incoming_trips"))
        .withColumnRenamed("dst", "id")
)

# Combine with vertices and compute net flow
flow_balance = (
    gbsUnique.vertices
        .select("id", "landmark", "dockcount")
        .join(outgoing, on="id", how="left")
        .join(incoming, on="id", how="left")
        .fillna({"outgoing_trips": 0, "incoming_trips": 0})
        .withColumn("total_trips", F.col("incoming_trips") + F.col("outgoing_trips"))
        .withColumn("net_flow", F.col("incoming_trips") - F.col("outgoing_trips"))
        .withColumn(
            "role",
            F.when(F.col("net_flow") > 0, "sink")
             .when(F.col("net_flow") < 0, "source")
             .otherwise("balanced")
        )
)

print("Top 10 source stations (net exporters of trips):")
(
    flow_balance
        .orderBy(F.col("net_flow").asc())  # most negative first
        .select("id", "landmark", "dockcount", "outgoing_trips", "incoming_trips", "net_flow", "role")
        .limit(10)
        .show(truncate=False)
)

print("Top 10 sink stations (net importers of trips):")
(
    flow_balance
        .orderBy(F.col("net_flow").desc())  # most positive first
        .select("id", "landmark", "dockcount", "outgoing_trips", "incoming_trips", "net_flow", "role")
        .limit(10)
        .show(truncate=False)
)

print("Summary of roles:")
(
    flow_balance
        .groupBy("role")
        .agg(
            F.count("*").alias("num_stations"),
            F.avg("net_flow").alias("avg_net_flow"),
            F.avg("total_trips").alias("avg_total_trips")
        )
        .orderBy("role")
        .show()
)


Top 10 source stations (net exporters of trips):


+---------------------------------------------+-------------+---------+--------------+--------------+--------+------+
|id                                           |landmark     |dockcount|outgoing_trips|incoming_trips|net_flow|role  |
+---------------------------------------------+-------------+---------+--------------+--------------+--------+------+
|Grant Avenue at Columbus Avenue              |San Francisco|15       |8337          |4319          |-4018   |source|
|2nd at Folsom                                |San Francisco|19       |7999          |4727          |-3272   |source|
|Powell at Post (Union Square)                |San Francisco|19       |6425          |4134          |-2291   |source|
|Beale at Market                              |San Francisco|19       |8359          |6330          |-2029   |source|
|Market at 10th                               |San Francisco|27       |11885         |10220         |-1665   |source|
|Temporary Transbay Terminal (Howard at Beale)|San Franc

+---------------------------------------------+-------------+---------+--------------+--------------+--------+----+
|id                                           |landmark     |dockcount|outgoing_trips|incoming_trips|net_flow|role|
+---------------------------------------------+-------------+---------+--------------+--------------+--------+----+
|San Francisco Caltrain (Townsend at 4th)     |San Francisco|19       |26304         |34810         |8506    |sink|
|Market at Sansome                            |San Francisco|27       |11431         |13916         |2485    |sink|
|Townsend at 7th                              |San Francisco|15       |13752         |15422         |1670    |sink|
|2nd at Townsend                              |San Francisco|27       |14026         |15463         |1437    |sink|
|Embarcadero at Vallejo                       |San Francisco|15       |5037          |6146          |1109    |sink|
|Embarcadero at Sansome                       |San Francisco|15       |1

+--------+------------+------------------+-----------------+
|    role|num_stations|      avg_net_flow|  avg_total_trips|
+--------+------------+------------------+-----------------+
|balanced|           4|               0.0|            506.0|
|    sink|          31| 703.1935483870968|12794.41935483871|
|  source|          35|-649.0857142857143|8400.114285714286|
+--------+------------+------------------+-----------------+



### Exercise 7 – Network robustness under hub failures

In this exercise, you will explore how robust the bike network is to the loss of its most important hubs. Start by identifying a small set of central stations (for example, using a centrality or hub measure from a previous exercise) and then simulate their failure by removing them and their incident edges from the graph. Compare the connectivity of the original and modified networks: examine changes in the size of the largest connected component, the number of isolated stations, and the distribution of component sizes. Use these comparisons to argue whether the system is resilient to the loss of key hubs or whether a few stations are critical for maintaining overall connectivity.



In [31]:

top_hubs_df = (
    pr.vertices
      .orderBy(F.col("pagerank").desc())
      .select("id", "landmark", "dockcount", "pagerank")
      .limit(3)
)

top_hubs_df.show(truncate=False)

hub_ids = [row["id"] for row in top_hubs_df.collect()]

# 2. Build a graph with these hubs removed
vertices_removed = gbsUnique.vertices.filter(~F.col("id").isin(hub_ids))
edges_removed = (
    gbsUnique.edges
        .filter(
            (~F.col("src").isin(hub_ids)) &
            (~F.col("dst").isin(hub_ids))
        )
)

g_removed = GraphFrame(vertices_removed, edges_removed)

# 3. Connected components for original graph
cc_orig = gbsUnique.connectedComponents()
sizes_orig = (
    cc_orig
        .groupBy("component")
        .agg(F.count("*").alias("size"))
        .orderBy(F.col("size").desc())
)

giant_size_orig = sizes_orig.first()["size"]
num_components_orig = sizes_orig.count()
num_isolated_orig = sizes_orig.filter(F.col("size") == 1).count()

print("Original graph connectivity:")
print(f"  Total stations:          {gbsUnique.vertices.count()}")
print(f"  Number of components:    {num_components_orig}")
print(f"  Giant component size:    {giant_size_orig}")
print(f"  Number of isolated nodes:{num_isolated_orig}")

# 4. Connected components for graph with hubs removed
cc_removed = g_removed.connectedComponents()
sizes_removed = (
    cc_removed
        .groupBy("component")
        .agg(F.count("*").alias("size"))
        .orderBy(F.col("size").desc())
)

giant_size_removed = sizes_removed.first()["size"] if sizes_removed.count() > 0 else 0
num_components_removed = sizes_removed.count()
num_isolated_removed = sizes_removed.filter(F.col("size") == 1).count()

print("\nGraph connectivity after removing top hubs:")
print(f"  Total stations (remaining):   {g_removed.vertices.count()}")
print(f"  Number of components:         {num_components_removed}")
print(f"  Giant component size:         {giant_size_removed}")
print(f"  Number of isolated nodes:     {num_isolated_removed}")

print("\nComponent size distribution after removal:")
sizes_removed.show()


+----------------------------------------+-------------+---------+------------------+
|id                                      |landmark     |dockcount|pagerank          |
+----------------------------------------+-------------+---------+------------------+
|Mountain View Caltrain Station          |Mountain View|23       |1.2174633693589552|
|California Ave Caltrain Station         |Palo Alto    |15       |1.2105684295459649|
|San Francisco Caltrain (Townsend at 4th)|San Francisco|19       |1.188219541479926 |
+----------------------------------------+-------------+---------+------------------+



Original graph connectivity:
  Total stations:          70
  Number of components:    3
  Giant component size:    68
  Number of isolated nodes:2



Graph connectivity after removing top hubs:
  Total stations (remaining):   67
  Number of components:         3
  Giant component size:         65
  Number of isolated nodes:     2

Component size distribution after removal:
+---------+----+
|component|size|
+---------+----+
|        0|  65|
|       65|   1|
|       35|   1|
+---------+----+

